# Lab 04-04b: Persistent Memory and State

**Module:** 04 — Assembling and Deploying Applications  
**Exam Weight:** ~10 questions across Module 04 (22% of exam)  
**UI Sections:** Catalog, SQL Editor  
**Difficulty:** Intermediate  
**Estimated Time:** 60–90 minutes

---

## Learning Objectives

- Understand **why** LLM-based agents need external persistent memory
- Implement five distinct memory patterns using Delta tables and Unity Catalog:
  1. **Buffer Memory** — raw conversation history
  2. **Summary Memory** — compressed history for long conversations
  3. **Structured Memory** — key-value fact extraction (scratchpad)
  4. **Feature Tables as Memory** — entity-attribute lookup
  5. **Checkpointing Agent State** — resumable long-running tasks
- Choose the right memory pattern for a given scenario (exam skill)

---

## Exam Objectives Covered

- **Configure a persistent datastore to store and retrieve intermediate memory or structured information**

---

## What Is Persistent Memory and Why Do Agents Need It?

Large Language Models are **stateless**. Every single API call starts from scratch — the model has zero recollection of anything you said in previous turns. This is fundamentally different from how humans have conversations, where we naturally build up context over time.

When you chat with an LLM through a UI like ChatGPT or Databricks Playground, it *looks* like the model remembers your conversation. But behind the scenes, the application is re-sending the entire conversation history with every request. That works fine for short chats, but it creates real problems for production agents:

| Problem | Why It Matters |
|---------|---------------|
| **Context window limits** | Models have a maximum token count (e.g., 128K tokens). Long conversations eventually exceed this. |
| **Cost** | You pay per token. Re-sending 50 previous messages every turn gets expensive fast. |
| **Cross-session continuity** | If a user comes back tomorrow, their in-memory history is gone. |
| **Multi-step agent workflows** | An agent that runs 20 tool calls might fail at step 15. Without checkpoints, it starts over. |
| **Structured facts** | Raw conversation text is a poor way to store "the customer's name is Alice and her order ID is 12345." |

**Persistent memory** solves all of these by writing conversation state to an external datastore — in Databricks, that means **Delta tables** managed by **Unity Catalog**. The agent reads from these tables at the start of each turn and writes back at the end.

This lab walks through five patterns for doing this, from simplest to most sophisticated.

---

## Mental Model

> **Think of an LLM like a brilliant consultant with amnesia.** Every time you call them, they have forgotten everything from your last conversation. They walk into the room, sit down, and say "How can I help you?" — with absolutely no memory of the two hours you spent together yesterday.
>
> **Persistent memory is the notepad on their desk.** Before each meeting, an assistant slides them a folder with:
> - The last few messages you exchanged (buffer memory)
> - A one-paragraph summary of your longer history (summary memory)
> - A card with key facts: your name, your account number, your preferences (structured memory)
> - A company directory they can look up any customer in (feature table)
> - A checklist of where they left off on a multi-day project (checkpoint)
>
> The consultant is still brilliant. They just need that folder to avoid asking you the same questions over and over.

---

## Exam Alert: Common Traps

The exam tests whether you understand *how* and *where* memory is persisted — not just that it exists. Watch for these:

| Trap / Distractor | Why It Is Wrong | Correct Answer |
|-------------------|----------------|----------------|
| "The LLM remembers previous turns automatically" | LLMs are stateless; memory is an application-layer concern, not a model feature | Memory must be explicitly stored and re-injected into the prompt |
| "Store conversation history in the model's weights" | Fine-tuning is not memory; it changes general behavior, not per-session state | Use an external datastore (Delta table) for per-session state |
| "Use a Python dictionary for persistent memory" | A Python dict lives in RAM and is lost when the cluster restarts or the session ends | Use a Delta table for durability across sessions and restarts |
| "Summary memory replaces buffer memory" | They serve different purposes and can coexist | Buffer = recent turns verbatim; Summary = compressed older context. Use both together. |
| "Feature tables and Delta tables are the same thing" | Feature tables have additional semantics (primary key, online serving, point-in-time lookup) | A feature table IS a Delta table with extra metadata and constraints |

---

## Prerequisites and UI Navigation

### What You Need
- A Databricks workspace with Unity Catalog enabled
- Access to the `workspace` catalog (or substitute your own catalog name)
- A cluster running **DBR 14.3 LTS** or later (for Delta table features)
- Access to a Foundation Model API endpoint (e.g., `databricks-meta-llama-3-1-70b-instruct`)

### UI Paths Used in This Lab

| Action | Navigation Path |
|--------|-----------------|
| View/create schema | **Catalog** (left sidebar) > `workspace` > `genai_labs` |
| Query Delta tables | **SQL Editor** (left sidebar) > New Query |
| View feature tables | **Catalog** > select table > "Feature Table" badge |
| Browse table data | **Catalog** > select table > **Sample Data** tab |

### Schema Setup
All tables in this lab live under `workspace.genai_labs`. We will create this schema if it does not already exist.

---

## Setup

In [ ]:
# Install required packages (only needed if not pre-installed on your cluster)
%pip install langchain-community -q
dbutils.library.restartPython()  # Restart Python to pick up new packages

In [ ]:
# ---- Imports ----
from pyspark.sql import SparkSession
from pyspark.sql import functions as F           # PySpark SQL functions (col, lit, current_timestamp, etc.)
from pyspark.sql.types import (                  # Schema definition types
    StructType, StructField, StringType, TimestampType, IntegerType
)
from datetime import datetime                    # Python datetime for timestamps
import json                                      # For serializing agent state to JSON
import mlflow                                    # MLflow for model/experiment tracking

# ---- Spark Session ----
# In Databricks notebooks, `spark` is pre-initialized. This line is a safety net.
spark = SparkSession.builder.getOrCreate()

# ---- Create the lab schema if it does not exist ----
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.genai_labs")

print("Setup complete. Schema workspace.genai_labs is ready.")

### Expected Output

```
Setup complete. Schema workspace.genai_labs is ready.
```

### What Just Happened

We installed `langchain-community` (which provides memory abstractions we will reference conceptually), imported PySpark and utility libraries, and ensured the `workspace.genai_labs` schema exists in Unity Catalog. All tables we create in this lab will live under this schema so they are governed, discoverable, and queryable from the SQL Editor.

---

## Step 1: Buffer Memory — Store Recent Conversation History

### What Is Buffer Memory?

Buffer memory is the simplest pattern: **save every message in the conversation to a table, and read them all back at the start of each turn.** It is called "buffer" because you are buffering (collecting) the full, unmodified conversation.

### When to Use It
- Short to medium conversations (under ~50 turns)
- When you need exact verbatim history (e.g., for auditing or compliance)
- When token cost is not a primary concern

### How It Works in Databricks
1. Create a Delta table with columns: `session_id`, `role`, `content`, `timestamp`
2. After each user message and assistant response, append rows to the table
3. At the start of each turn, read all rows for the current `session_id`, ordered by timestamp
4. Inject those messages into the LLM prompt as conversation history

The `session_id` column is critical — it lets you maintain separate histories for different users or conversations, all in the same table.

In [ ]:
# ---- Step 1a: Create the buffer memory table ----
# This Delta table stores every message in every conversation.
# Think of it as an append-only chat log.

spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.genai_labs.conversation_buffer (
        session_id   STRING     COMMENT 'Unique identifier for each conversation session',
        role         STRING     COMMENT 'Who sent this message: user, assistant, or system',
        content      STRING     COMMENT 'The actual message text',
        timestamp    TIMESTAMP  COMMENT 'When this message was sent'
    )
    USING DELTA
    COMMENT 'Buffer memory: stores full conversation history for agent sessions'
""")

print("Table workspace.genai_labs.conversation_buffer created (or already exists).")

### Expected Output

```
Table workspace.genai_labs.conversation_buffer created (or already exists).
```

### What Just Happened

We created a Delta table with four columns. The `session_id` lets us partition conversations — one table serves all users and all sessions. The `role` column follows the OpenAI/Databricks convention of `user`, `assistant`, and `system` roles. The `USING DELTA` clause ensures we get ACID transactions, time travel, and all the other Delta Lake guarantees.

You can verify this table exists by navigating to **Catalog > main > genai_labs > conversation_buffer**.

In [ ]:
# ---- Step 1b: Insert messages into the buffer ----
# Simulate a short conversation between a user and an assistant.
# In production, your agent framework would call this after each turn.

from pyspark.sql import Row

# Define a helper function to append a message to the buffer
def append_to_buffer(session_id: str, role: str, content: str):
    """
    Appends a single message to the conversation_buffer Delta table.
    
    Parameters:
        session_id: Unique ID for this conversation session
        role:       'user', 'assistant', or 'system'
        content:    The message text
    """
    # Create a single-row DataFrame with the message
    row = spark.createDataFrame([{
        "session_id": session_id,
        "role": role,
        "content": content,
        "timestamp": datetime.now()       # Current time as the message timestamp
    }])
    # Append (not overwrite) to the Delta table
    row.write.mode("append").saveAsTable("workspace.genai_labs.conversation_buffer")


# Simulate a 3-turn conversation
session = "session-abc-123"

append_to_buffer(session, "user",      "Hi! I need help tracking my order.")
append_to_buffer(session, "assistant",  "Of course! Could you provide your order ID?")
append_to_buffer(session, "user",      "It's ORD-98765.")
append_to_buffer(session, "assistant",  "Thanks! Order ORD-98765 shipped yesterday and should arrive by Friday.")
append_to_buffer(session, "user",      "Can I change the delivery address?")

print(f"Inserted 5 messages for session '{session}'.")

### Expected Output

```
Inserted 5 messages for session 'session-abc-123'.
```

### What Just Happened

We defined a reusable `append_to_buffer()` function and used it to insert five messages — a back-and-forth between a user and an assistant. Each call creates a one-row DataFrame and appends it to the Delta table. In a real agent, you would call this function after every user message and every assistant response.

In [ ]:
# ---- Step 1c: Read the buffer back — this is what gets injected into the prompt ----
# At the start of each new turn, the agent reads the full history for this session.

def get_buffer_history(session_id: str) -> list:
    """
    Retrieves the full conversation history for a session, ordered by time.
    Returns a list of dicts in the format LLMs expect: [{"role": ..., "content": ...}]
    """
    # Read from Delta, filter by session, order chronologically
    history_df = (
        spark.table("workspace.genai_labs.conversation_buffer")
        .filter(F.col("session_id") == session_id)   # Only this session
        .orderBy(F.col("timestamp").asc())            # Oldest first
        .select("role", "content")                    # Only the fields the LLM needs
    )
    # Convert to a list of Python dicts
    return [row.asDict() for row in history_df.collect()]


# Retrieve and display the history
history = get_buffer_history("session-abc-123")

print(f"Retrieved {len(history)} messages:\n")
for msg in history:
    print(f"  [{msg['role']:>9}]  {msg['content']}")

### Expected Output

```
Retrieved 5 messages:

  [     user]  Hi! I need help tracking my order.
  [assistant]  Of course! Could you provide your order ID?
  [     user]  It's ORD-98765.
  [assistant]  Thanks! Order ORD-98765 shipped yesterday and should arrive by Friday.
  [     user]  Can I change the delivery address?
```

### What Just Happened

We read back the full conversation for session `session-abc-123`, ordered by timestamp. The output is a list of `{"role": ..., "content": ...}` dictionaries — exactly the format you would pass to an LLM's `messages` parameter. This is the core loop of buffer memory: write after each turn, read before each turn.

**Key insight for the exam:** The conversation history is stored *externally* in a Delta table. If the cluster restarts, the notebook is closed, or the user comes back next week, the history is still there. A Python list or dictionary would have been lost.

---

## Step 2: Summary Memory — Compress Older History to Save Tokens

### The Problem with Buffer Memory

Buffer memory works great for short conversations. But what happens when a conversation goes on for 200 turns? You would be sending thousands of tokens of history with every single LLM call. That is:
- **Expensive** — you pay per token
- **Slow** — more input tokens means higher latency
- **Risky** — you might exceed the model's context window

### The Solution: Summary Memory

Summary memory keeps the **last N messages verbatim** (so the agent has recent context) and replaces everything older with a **compressed summary**. The summary is generated by an LLM call and stored in the Delta table.

Think of it like meeting notes: you remember the last 10 minutes of discussion in detail, but for everything before that, you rely on the notes someone wrote on the whiteboard.

### The Pattern
1. Keep the most recent `N` messages as-is (buffer)
2. Take all messages older than that and feed them to an LLM with the prompt: "Summarize this conversation so far"
3. Store the summary in a `conversation_summaries` table
4. On the next turn, load: `[system summary] + [last N messages]`

In [ ]:
# ---- Step 2a: Create a table to store conversation summaries ----

spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.genai_labs.conversation_summaries (
        session_id       STRING     COMMENT 'Links to the same session_id in conversation_buffer',
        summary_text     STRING     COMMENT 'LLM-generated summary of older messages',
        messages_covered INT        COMMENT 'How many messages this summary covers',
        created_at       TIMESTAMP  COMMENT 'When this summary was generated'
    )
    USING DELTA
    COMMENT 'Summary memory: compressed conversation context for long sessions'
""")

print("Table workspace.genai_labs.conversation_summaries created (or already exists).")

### Expected Output

```
Table workspace.genai_labs.conversation_summaries created (or already exists).
```

### What Just Happened

We created a second Delta table specifically for summaries. Each row captures a summary for a given session, how many messages it covers (so we know which messages are already summarized), and when it was created. A session can have multiple summary rows over time as the conversation grows — each new summary replaces or extends the previous one.

In [ ]:
# ---- Step 2b: Generate a summary of older messages using an LLM ----
# This function calls the Databricks Foundation Model API to summarize conversation history.

import mlflow.deployments

def summarize_conversation(messages: list) -> str:
    """
    Takes a list of message dicts and asks an LLM to produce a concise summary.
    Uses the Databricks Foundation Model API (pay-per-token).
    
    Parameters:
        messages: List of {"role": ..., "content": ...} dicts to summarize
    Returns:
        A string containing the summary
    """
    # Get a client for the Databricks model serving endpoint
    client = mlflow.deployments.get_deploy_client("databricks")
    
    # Format the messages into a readable block for the summarizer
    conversation_text = "\n".join(
        f"{m['role']}: {m['content']}" for m in messages
    )
    
    # Call the LLM with a summarization prompt
    response = client.predict(
        endpoint="databricks-meta-llama-3-1-70b-instruct",  # Pay-per-token endpoint
        inputs={
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "You are a conversation summarizer. Produce a concise summary "
                        "of the following conversation. Focus on key facts, decisions, "
                        "and any unresolved questions. Keep it under 100 words."
                    )
                },
                {
                    "role": "user",
                    "content": f"Summarize this conversation:\n\n{conversation_text}"
                }
            ],
            "max_tokens": 200,         # Keep the summary short
            "temperature": 0.0          # Deterministic output for consistency
        }
    )
    
    # Extract the summary text from the response
    return response.choices[0].message.content


# ---- Step 2c: Build summary memory for our session ----
# Strategy: keep the last 2 messages verbatim, summarize everything older

RECENT_WINDOW = 2  # Number of recent messages to keep verbatim

# Get all messages for this session
all_messages = get_buffer_history("session-abc-123")

if len(all_messages) > RECENT_WINDOW:
    # Split into "old" (to summarize) and "recent" (to keep verbatim)
    old_messages = all_messages[:-RECENT_WINDOW]
    recent_messages = all_messages[-RECENT_WINDOW:]
    
    # Generate a summary of the older messages
    summary = summarize_conversation(old_messages)
    
    # Save the summary to the summaries table
    summary_row = spark.createDataFrame([{
        "session_id":       "session-abc-123",
        "summary_text":     summary,
        "messages_covered": len(old_messages),
        "created_at":       datetime.now()
    }])
    summary_row.write.mode("append").saveAsTable("workspace.genai_labs.conversation_summaries")
    
    print("=== SUMMARY (replaces older messages) ===")
    print(summary)
    print("\n=== RECENT MESSAGES (kept verbatim) ===")
    for msg in recent_messages:
        print(f"  [{msg['role']:>9}]  {msg['content']}")
else:
    print("Conversation is short enough — no summary needed yet.")

### Expected Output

```
=== SUMMARY (replaces older messages) ===
The user asked for help tracking order ORD-98765. The assistant confirmed 
the order shipped yesterday and should arrive by Friday.

=== RECENT MESSAGES (kept verbatim) ===
  [assistant]  Thanks! Order ORD-98765 shipped yesterday and should arrive by Friday.
  [     user]  Can I change the delivery address?
```

(Your exact summary text will vary because it is LLM-generated.)

### What Just Happened

We split the conversation into two parts:
1. **Old messages** (the first 3) were sent to an LLM to be compressed into a short summary
2. **Recent messages** (the last 2) were kept word-for-word

The summary was saved to `workspace.genai_labs.conversation_summaries`. On the next turn, instead of sending all 5+ messages, the agent would send: `[system: <summary>] + [last 2 messages]`. This dramatically reduces token usage for long conversations.

**Exam tip:** Summary memory does NOT delete the original messages. The full buffer is still in `conversation_buffer` for auditing. The summary is an *optimization layer* on top.

In [ ]:
# ---- Step 2d: Build the final prompt using summary + recent messages ----
# This is what you would actually send to the LLM on the next turn.

def build_prompt_with_summary(session_id: str, recent_window: int = 2) -> list:
    """
    Constructs an LLM-ready message list using:
      1. The latest summary (if one exists) as a system message
      2. The most recent N messages verbatim
    
    Returns:
        List of {"role": ..., "content": ...} dicts ready for the LLM
    """
    prompt_messages = []
    
    # Check for an existing summary
    summary_df = (
        spark.table("workspace.genai_labs.conversation_summaries")
        .filter(F.col("session_id") == session_id)
        .orderBy(F.col("created_at").desc())      # Most recent summary first
        .limit(1)                                  # Only the latest
    )
    
    summary_rows = summary_df.collect()
    if summary_rows:
        # Inject the summary as a system message at the top
        prompt_messages.append({
            "role": "system",
            "content": f"Summary of earlier conversation: {summary_rows[0]['summary_text']}"
        })
    
    # Get the most recent messages from the buffer
    all_messages = get_buffer_history(session_id)
    recent = all_messages[-recent_window:] if len(all_messages) > recent_window else all_messages
    prompt_messages.extend(recent)
    
    return prompt_messages


# Build and display the prompt
final_prompt = build_prompt_with_summary("session-abc-123", recent_window=2)

print("Final prompt for the LLM:\n")
for msg in final_prompt:
    print(f"  [{msg['role']:>9}]  {msg['content'][:100]}..." 
          if len(msg['content']) > 100 
          else f"  [{msg['role']:>9}]  {msg['content']}")

### Expected Output

```
Final prompt for the LLM:

  [   system]  Summary of earlier conversation: The user asked for help tracking order ORD-98765...
  [assistant]  Thanks! Order ORD-98765 shipped yesterday and should arrive by Friday.
  [     user]  Can I change the delivery address?
```

### What Just Happened

We assembled the final prompt that would go to the LLM. Instead of 5 messages (which would grow to 50 or 500 in a real conversation), we sent just 3: one system message with the compressed summary, plus the two most recent messages. The LLM has enough context to continue the conversation intelligently without the full verbatim history.

---

## Step 3: Structured Memory — Key-Value Fact Extraction

### Beyond Chat Logs

Buffer and summary memory store *conversation text*. But conversations contain **structured facts** that are more useful when extracted and stored separately:

- "My name is Alice" -> `{"customer_name": "Alice"}`
- "Order ID is ORD-98765" -> `{"order_id": "ORD-98765"}`
- "I prefer email notifications" -> `{"notification_preference": "email"}`

Structured memory is like a **scratchpad** that the agent uses between tool calls. Instead of parsing through 50 messages to find the customer's name, the agent just looks up `customer_name` in the structured memory table.

### Why This Matters
- **Faster lookups** — no need to search through conversation text
- **Cross-session persistence** — facts survive beyond a single conversation
- **Tool integration** — downstream tools can read structured facts directly (e.g., an order-tracking API needs the order ID, not a paragraph of chat)
- **Deduplication** — the latest value for a key overwrites the old one

In [ ]:
# ---- Step 3a: Create the structured memory table ----
# This is a key-value store backed by Delta Lake.
# Each row is one fact: a (session_id, key) -> value mapping.

spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.genai_labs.structured_memory (
        session_id   STRING     COMMENT 'Links to the conversation session',
        fact_key     STRING     COMMENT 'The name of the fact, e.g. customer_name, order_id',
        fact_value   STRING     COMMENT 'The extracted value, e.g. Alice, ORD-98765',
        updated_at   TIMESTAMP  COMMENT 'When this fact was last updated'
    )
    USING DELTA
    COMMENT 'Structured memory: key-value facts extracted from conversations'
""")

print("Table workspace.genai_labs.structured_memory created (or already exists).")

### Expected Output

```
Table workspace.genai_labs.structured_memory created (or already exists).
```

### What Just Happened

We created a key-value store as a Delta table. Each row represents one fact about a session. The `fact_key` is the name (like `customer_name`) and `fact_value` is the extracted value (like `Alice`). We will use `MERGE` operations to upsert — insert new facts or update existing ones.

In [ ]:
# ---- Step 3b: Write structured facts using MERGE (upsert) ----
# MERGE is the key operation here: if the fact already exists, update it.
# If it is new, insert it. This prevents duplicate rows for the same fact.

def upsert_fact(session_id: str, key: str, value: str):
    """
    Inserts or updates a single fact in structured memory.
    Uses Delta Lake MERGE to handle the upsert atomically.
    
    Parameters:
        session_id: The conversation session this fact belongs to
        key:        Fact name (e.g., 'customer_name')
        value:      Fact value (e.g., 'Alice')
    """
    # Create a temporary view with the new fact
    new_fact = spark.createDataFrame([{
        "session_id": session_id,
        "fact_key":   key,
        "fact_value":  value,
        "updated_at": datetime.now()
    }])
    new_fact.createOrReplaceTempView("new_fact")
    
    # MERGE: match on session_id + fact_key
    # If matched -> update the value and timestamp
    # If not matched -> insert a new row
    spark.sql("""
        MERGE INTO workspace.genai_labs.structured_memory AS target
        USING new_fact AS source
        ON target.session_id = source.session_id 
           AND target.fact_key = source.fact_key
        WHEN MATCHED THEN UPDATE SET
            target.fact_value = source.fact_value,
            target.updated_at = source.updated_at
        WHEN NOT MATCHED THEN INSERT *
    """)


# ---- Extract and store facts from our conversation ----
# In production, you would use an LLM to extract these automatically.
# Here we do it manually to focus on the storage pattern.

session = "session-abc-123"

upsert_fact(session, "customer_intent",          "track order")
upsert_fact(session, "order_id",                 "ORD-98765")
upsert_fact(session, "order_status",             "shipped")
upsert_fact(session, "estimated_delivery",       "Friday")
upsert_fact(session, "pending_request",          "change delivery address")

print(f"Upserted 5 structured facts for session '{session}'.")

### Expected Output

```
Upserted 5 structured facts for session 'session-abc-123'.
```

### What Just Happened

We used Delta Lake's `MERGE` statement to upsert facts. This is critical: if the customer later says "Actually, my order ID is ORD-99999," calling `upsert_fact(session, "order_id", "ORD-99999")` will **update** the existing row instead of creating a duplicate. The `MERGE` pattern gives us a clean, deduplicated key-value store on top of Delta Lake.

**Exam connection:** The exam may ask how to store structured information that persists across agent turns. The answer is a Delta table with MERGE for upsert — not a Python dict (volatile) and not the conversation buffer (unstructured).

In [ ]:
# ---- Step 3c: Read structured facts back — the agent's scratchpad ----

def get_structured_facts(session_id: str) -> dict:
    """
    Retrieves all structured facts for a session as a Python dictionary.
    This is the agent's "scratchpad" — quick access to key information.
    
    Returns:
        Dict like {"customer_name": "Alice", "order_id": "ORD-98765", ...}
    """
    facts_df = (
        spark.table("workspace.genai_labs.structured_memory")
        .filter(F.col("session_id") == session_id)
        .select("fact_key", "fact_value")
    )
    # Convert to a simple key-value dict
    return {row["fact_key"]: row["fact_value"] for row in facts_df.collect()}


# Read and display
facts = get_structured_facts("session-abc-123")

print("Structured facts for session-abc-123:\n")
for key, value in sorted(facts.items()):
    print(f"  {key:25s} = {value}")

### Expected Output

```
Structured facts for session-abc-123:

  customer_intent           = track order
  estimated_delivery        = Friday
  order_id                  = ORD-98765
  order_status              = shipped
  pending_request           = change delivery address
```

### What Just Happened

We read back the structured facts as a Python dictionary. An agent can now use this to:
- Pass `order_id` directly to an order-tracking API tool
- Include key facts in the system prompt: "The customer is tracking order ORD-98765, which shipped and arrives Friday. They want to change the delivery address."
- Avoid re-asking the user for information it already has

This is fundamentally different from buffer memory. Buffer memory stores *what was said*. Structured memory stores *what was learned*.

---

## Step 4: Feature Tables as Memory — Entity-Attribute Lookup

### From Session Memory to Entity Memory

The memory patterns so far are **session-scoped** — they track information about a single conversation. But what about information that persists *across all conversations* for a given entity (e.g., a customer, a product, an account)?

This is where **Feature Tables** come in. A feature table is a Delta table with extra semantics:
- It has a **primary key** (e.g., `customer_id`)
- It can be served via **online tables** for low-latency lookups
- It integrates with Databricks **Feature Store** and **Feature Serving**
- It supports **point-in-time lookups** for training ML models

### Comparison: When to Use What

| Use Case | Storage | Example |
|----------|---------|--------|
| What was said in this conversation | `conversation_buffer` (Delta) | "User said: My order is ORD-98765" |
| Key facts from this conversation | `structured_memory` (Delta) | `order_id = ORD-98765` |
| Persistent attributes about a customer | Feature table | `customer_id=C001 -> name=Alice, tier=Gold, region=US-West` |

### UI Navigation

To view feature tables:
1. Go to **Catalog** in the left sidebar
2. Navigate to your catalog > schema > table
3. Feature tables are marked with a **"Feature Table"** badge
4. You can also find them under **Machine Learning > Feature Store** (legacy UI)

In [ ]:
# ---- Step 4a: Create a feature table for customer profiles ----
# This table stores persistent entity attributes that any agent can look up.
# The primary key is customer_id — each customer has exactly one row.

from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()  # Client for the Databricks Feature Store

# Define sample customer data
customer_data = spark.createDataFrame([
    {"customer_id": "C001", "name": "Alice",   "tier": "Gold",   "region": "US-West",  "preferred_channel": "email"},
    {"customer_id": "C002", "name": "Bob",     "tier": "Silver", "region": "US-East",  "preferred_channel": "sms"},
    {"customer_id": "C003", "name": "Charlie", "tier": "Gold",   "region": "EU-North", "preferred_channel": "email"},
])

# Create (or overwrite) the feature table with customer_id as the primary key
# This registers the table in Unity Catalog AND marks it as a feature table
fe.create_table(
    name="workspace.genai_labs.customer_profiles",    # Full Unity Catalog path
    primary_keys=["customer_id"],                # Primary key for lookups
    df=customer_data,                            # Initial data
    description="Customer profiles for agent lookups. Primary key: customer_id."
)

print("Feature table workspace.genai_labs.customer_profiles created.")
print("Navigate to Catalog > main > genai_labs > customer_profiles to see the Feature Table badge.")

### Expected Output

```
Feature table workspace.genai_labs.customer_profiles created.
Navigate to Catalog > main > genai_labs > customer_profiles to see the Feature Table badge.
```

### What Just Happened

We used the `FeatureEngineeringClient` to create a feature table. Under the hood, this is still a Delta table — but it has additional metadata:
- The `customer_id` column is registered as the **primary key**
- The table appears with a **Feature Table** badge in the Catalog UI
- It can be published to an **online table** for real-time serving (sub-millisecond lookups)

**Why not just use a regular Delta table?** You could. But feature tables give you primary key enforcement, integration with Feature Serving endpoints, and a clear semantic signal that this table is meant for entity lookups — not just data storage.

In [ ]:
# ---- Step 4b: Look up a customer profile — what an agent does mid-conversation ----
# When the agent learns that the customer is "C001", it looks up their profile.

def lookup_customer(customer_id: str) -> dict:
    """
    Retrieves a customer profile from the feature table.
    This is what an agent tool would call during a conversation.
    
    Parameters:
        customer_id: The primary key to look up
    Returns:
        A dict with the customer's attributes, or None if not found
    """
    # Simple Delta table read with a filter on the primary key
    result = (
        spark.table("workspace.genai_labs.customer_profiles")
        .filter(F.col("customer_id") == customer_id)
        .collect()
    )
    # Return the first (and only) matching row as a dict, or None
    return result[0].asDict() if result else None


# Look up customer C001
profile = lookup_customer("C001")

if profile:
    print("Customer profile found:\n")
    for key, value in profile.items():
        print(f"  {key:20s} = {value}")
else:
    print("Customer not found.")

### Expected Output

```
Customer profile found:

  customer_id          = C001
  name                 = Alice
  tier                 = Gold
  region               = US-West
  preferred_channel    = email
```

### What Just Happened

The agent looked up customer C001 and instantly got their profile: name, tier, region, and preferred channel. This information can be injected into the system prompt ("You are helping Alice, a Gold-tier customer in US-West who prefers email") or used by tools ("Send a notification via email because that is the customer's preferred channel").

**Key distinction for the exam:**
- **Structured memory** (`structured_memory` table) = facts extracted *during this conversation*. Session-scoped.
- **Feature table** (`customer_profiles` table) = pre-existing entity attributes. Cross-session, cross-agent.

---

## Step 5: Checkpointing Agent State — Resumable Long-Running Tasks

### The Problem: Long-Running Agents Fail

Some agent workflows are not simple question-answer conversations. They are multi-step processes that can take minutes or even hours:

- Analyzing 50 documents and writing a report
- Processing a batch of customer tickets one by one
- Running a multi-step data pipeline with validation at each stage

If the agent fails at step 15 of 20, you do not want to start over from step 1. **Checkpointing** saves the agent's progress so it can resume from where it left off.

### How It Works
1. Create a `checkpoints` table with columns: `run_id`, `step`, `state_json`, `timestamp`
2. After completing each major step, the agent writes a checkpoint
3. The `state_json` column stores a JSON blob with everything needed to resume: intermediate results, which items have been processed, partial outputs, etc.
4. On restart, the agent reads the latest checkpoint for its `run_id` and picks up from there

This is analogous to a video game save system — you save at checkpoints so you do not replay the whole game if you lose.

In [ ]:
# ---- Step 5a: Create the checkpoint table ----

spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.genai_labs.agent_checkpoints (
        run_id      STRING     COMMENT 'Unique identifier for this agent run (e.g., UUID)',
        step        INT        COMMENT 'Step number within the run (1, 2, 3, ...)',
        state_json  STRING     COMMENT 'JSON-serialized agent state at this checkpoint',
        status      STRING     COMMENT 'Status of this step: completed, failed, in_progress',
        timestamp   TIMESTAMP  COMMENT 'When this checkpoint was saved'
    )
    USING DELTA
    COMMENT 'Agent checkpoints: saves intermediate state for resumable long-running tasks'
""")

print("Table workspace.genai_labs.agent_checkpoints created (or already exists).")

### Expected Output

```
Table workspace.genai_labs.agent_checkpoints created (or already exists).
```

### What Just Happened

We created a checkpoint table. The key design choices:
- `run_id` identifies a specific execution (so multiple runs do not interfere)
- `step` is an integer so you can easily find the latest completed step
- `state_json` is a flexible string column that holds JSON — it can store anything the agent needs to resume
- `status` tracks whether the step completed, failed, or is still in progress

In [ ]:
# ---- Step 5b: Simulate a multi-step agent with checkpointing ----
# This agent processes a list of customer tickets. After each ticket,
# it saves a checkpoint so it can resume if interrupted.

import uuid

def save_checkpoint(run_id: str, step: int, state: dict, status: str = "completed"):
    """
    Saves a checkpoint for a long-running agent task.
    
    Parameters:
        run_id: Unique identifier for this agent run
        step:   Step number (1-indexed)
        state:  Dict with any state the agent needs to resume
        status: 'completed', 'failed', or 'in_progress'
    """
    checkpoint = spark.createDataFrame([{
        "run_id":     run_id,
        "step":       step,
        "state_json": json.dumps(state),      # Serialize the state dict to JSON
        "status":     status,
        "timestamp":  datetime.now()
    }])
    checkpoint.write.mode("append").saveAsTable("workspace.genai_labs.agent_checkpoints")


def get_latest_checkpoint(run_id: str) -> dict:
    """
    Retrieves the latest completed checkpoint for a given run.
    Returns the deserialized state dict, or None if no checkpoints exist.
    """
    result = (
        spark.table("workspace.genai_labs.agent_checkpoints")
        .filter(
            (F.col("run_id") == run_id) & 
            (F.col("status") == "completed")   # Only completed steps
        )
        .orderBy(F.col("step").desc())          # Latest step first
        .limit(1)
        .collect()
    )
    if result:
        row = result[0]
        return {
            "step":  row["step"],
            "state": json.loads(row["state_json"])  # Deserialize JSON back to dict
        }
    return None


# ---- Simulate the agent processing tickets ----
run_id = str(uuid.uuid4())  # Generate a unique run ID
tickets = ["TICKET-001", "TICKET-002", "TICKET-003", "TICKET-004", "TICKET-005"]

print(f"Starting agent run: {run_id}")
print(f"Processing {len(tickets)} tickets...\n")

processed = []  # Track which tickets have been processed

for i, ticket in enumerate(tickets, start=1):
    # Simulate processing (in reality, this would call an LLM or tool)
    result = f"Resolved: {ticket} - issue categorized and response drafted"
    processed.append({"ticket": ticket, "result": result})
    
    # Save checkpoint after each ticket
    save_checkpoint(
        run_id=run_id,
        step=i,
        state={
            "processed_tickets": processed.copy(),   # What we have done so far
            "remaining_tickets": tickets[i:],        # What is left
            "total_tickets":     len(tickets)
        },
        status="completed"
    )
    print(f"  Step {i}: Processed {ticket} -> checkpoint saved")

print(f"\nAll {len(tickets)} tickets processed. Run complete.")

### Expected Output

```
Starting agent run: <uuid>
Processing 5 tickets...

  Step 1: Processed TICKET-001 -> checkpoint saved
  Step 2: Processed TICKET-002 -> checkpoint saved
  Step 3: Processed TICKET-003 -> checkpoint saved
  Step 4: Processed TICKET-004 -> checkpoint saved
  Step 5: Processed TICKET-005 -> checkpoint saved

All 5 tickets processed. Run complete.
```

### What Just Happened

The agent processed 5 tickets sequentially, saving a checkpoint after each one. Each checkpoint contains the full state: which tickets were processed (with their results) and which remain. If the agent had crashed after step 3, we could reload the checkpoint and resume from ticket 4.

In [ ]:
# ---- Step 5c: Demonstrate resuming from a checkpoint ----
# Imagine the agent crashed. We retrieve the last checkpoint and resume.

latest = get_latest_checkpoint(run_id)

if latest:
    print(f"Latest checkpoint: Step {latest['step']}")
    print(f"\nState at checkpoint:")
    print(f"  Processed: {len(latest['state']['processed_tickets'])} tickets")
    print(f"  Remaining: {latest['state']['remaining_tickets']}")
    
    # If there were remaining tickets, we would resume processing them here
    if latest['state']['remaining_tickets']:
        print(f"\n  -> Would resume from: {latest['state']['remaining_tickets'][0]}")
    else:
        print(f"\n  -> All tickets already processed. Nothing to resume.")
else:
    print("No checkpoint found — starting from scratch.")

### Expected Output

```
Latest checkpoint: Step 5

State at checkpoint:
  Processed: 5 tickets
  Remaining: []

  -> All tickets already processed. Nothing to resume.
```

### What Just Happened

We demonstrated the resume logic. The function `get_latest_checkpoint()` reads the checkpoint table, finds the most recent completed step for our run ID, and returns the deserialized state. In a real scenario where the agent crashed at step 3, this would return step 3's state with `remaining_tickets = ["TICKET-004", "TICKET-005"]`, and the agent would resume from TICKET-004.

**Exam insight:** Checkpointing is the answer when the exam asks about making long-running agent tasks resumable or fault-tolerant. The key elements are: unique `run_id`, incrementing `step` number, and a JSON state column.

---

## Step 6: Choosing the Right Memory Pattern — Decision Framework

Now that you have seen all five patterns, how do you decide which one to use? The exam will give you scenarios and ask you to pick the correct approach. Here is the decision framework:

### Decision Table

| Scenario | Memory Pattern | Why |
|----------|---------------|-----|
| Short chatbot conversation (< 20 turns) | **Buffer Memory** | Simple, low overhead, token count is manageable |
| Long customer support session (100+ turns) | **Summary Memory** (+ buffer for recent turns) | Compresses older history to save tokens while keeping recent context verbatim |
| Agent needs quick access to facts (order ID, customer name) | **Structured Memory** | Key-value lookup is faster and cleaner than searching chat logs |
| Agent needs to look up customer profile across all conversations | **Feature Table** | Entity-level attributes that persist across sessions, with online serving capability |
| Multi-step data processing job that might fail | **Checkpointing** | Saves progress so the agent can resume without reprocessing completed work |
| All of the above in a production agent | **Combine them** | A real agent often uses 2-3 patterns together |

### Common Combinations

In practice, production agents often use multiple memory patterns together:

1. **Customer support agent:**
   - Buffer memory for the current session's exact messages
   - Summary memory to compress long sessions
   - Structured memory to track extracted facts (order ID, issue type)
   - Feature table to look up the customer's profile and history

2. **Document processing agent:**
   - Checkpointing to track which documents have been processed
   - Structured memory to store extracted metadata from each document
   - Feature table to look up document categories or templates

3. **Multi-turn research agent:**
   - Buffer memory for the full conversation
   - Structured memory to accumulate findings across turns
   - Checkpointing if the research spans multiple sessions

In [ ]:
# ---- Step 6: Putting it all together — verify all tables exist ----
# Let us confirm all five tables are in Unity Catalog and peek at their contents.

tables = [
    "workspace.genai_labs.conversation_buffer",     # Step 1: Buffer memory
    "workspace.genai_labs.conversation_summaries",  # Step 2: Summary memory
    "workspace.genai_labs.structured_memory",       # Step 3: Structured memory
    "workspace.genai_labs.customer_profiles",       # Step 4: Feature table
    "workspace.genai_labs.agent_checkpoints",       # Step 5: Checkpointing
]

print("Memory tables in workspace.genai_labs:\n")
print(f"{'Table Name':45s} {'Row Count':>10s}  {'Pattern'}")
print("-" * 85)

patterns = ["Buffer Memory", "Summary Memory", "Structured Memory", 
            "Feature Table", "Checkpointing"]

for table, pattern in zip(tables, patterns):
    count = spark.table(table).count()         # Count rows in each table
    print(f"  {table:43s} {count:>10d}  {pattern}")

print("\nAll persistent memory tables are live in Unity Catalog.")
print("You can query any of them from the SQL Editor.")

### Expected Output

```
Memory tables in workspace.genai_labs:

Table Name                                     Row Count  Pattern
-------------------------------------------------------------------------------------
  workspace.genai_labs.conversation_buffer                   5  Buffer Memory
  workspace.genai_labs.conversation_summaries                1  Summary Memory
  workspace.genai_labs.structured_memory                     5  Structured Memory
  workspace.genai_labs.customer_profiles                     3  Feature Table
  workspace.genai_labs.agent_checkpoints                     5  Checkpointing

All persistent memory tables are live in Unity Catalog.
You can query any of them from the SQL Editor.
```

### What Just Happened

We verified that all five memory tables exist in Unity Catalog and have the expected number of rows. These tables are now persistent — they survive cluster restarts, notebook closures, and can be queried from the SQL Editor, used by other notebooks, or served via Feature Serving endpoints.

---

## Exam Tips and Traps

| # | Tip or Trap | Explanation |
|---|-------------|-------------|
| 1 | **Trap:** "Use model fine-tuning to give the LLM memory" | Fine-tuning changes the model's general behavior. It does NOT give per-session memory. Persistent memory requires external storage. |
| 2 | **Trap:** "Python variables persist across agent invocations" | They do not. Each serving endpoint request is stateless. Even notebook variables are lost on cluster restart. Use Delta tables. |
| 3 | **Tip:** If the question mentions "across sessions" or "user returns later" | The answer involves a persistent datastore (Delta table), not in-memory state. |
| 4 | **Tip:** If the question mentions "long conversation" or "token limit" | Summary memory is the answer — compress older history, keep recent turns verbatim. |
| 5 | **Trap:** "Feature tables are only for ML training" | Feature tables can also serve as entity-level memory for agents via online serving. They are not limited to traditional ML. |
| 6 | **Tip:** `MERGE INTO` is how you upsert structured facts | The exam expects you to know that MERGE handles insert-or-update atomically in Delta Lake. |
| 7 | **Tip:** A feature table IS a Delta table (with extra metadata) | Do not treat them as completely different things. A feature table adds primary key enforcement and serving capabilities on top of Delta. |

---

## Key Takeaways

### Core Concepts

- LLMs are **stateless** — they have no built-in memory between API calls
- **Persistent memory** is an application-layer pattern: write state to an external store after each turn, read it back before the next turn
- Five memory patterns exist, each for a different use case:
  - **Buffer** = full conversation log (simple, verbatim)
  - **Summary** = compressed older history + recent messages (token-efficient)
  - **Structured** = key-value facts extracted from conversation (scratchpad)
  - **Feature Table** = entity-level attributes across all sessions (customer profile)
  - **Checkpoint** = intermediate state for resumable long-running tasks (save game)

### Databricks-Specific

- All memory tables are **Delta tables** in **Unity Catalog** — governed, versioned, queryable
- Use `MERGE INTO` for upserting structured facts (atomic insert-or-update)
- Use `FeatureEngineeringClient.create_table()` to create feature tables with primary keys
- Feature tables can be served via **online tables** for low-latency agent lookups
- All tables are accessible from the **SQL Editor**, **Catalog** browser, and other notebooks

### Exam Essentials

- The exam objective is: "Configure a persistent datastore to store and retrieve intermediate memory or structured information"
- **Delta table** is the correct persistent datastore for all memory patterns on Databricks
- **Python dicts/lists** are NOT persistent — they are lost on restart
- **Fine-tuning** is NOT memory — it changes model behavior globally, not per-session
- Know when to use each pattern: buffer (short), summary (long), structured (facts), feature table (entities), checkpoint (resumable tasks)
- `MERGE` is the SQL operation for upsert; the exam expects you to recognize this

---

## Cleanup (Optional)

Uncomment and run the following cell to remove the tables created in this lab. Only do this if you are done experimenting and do not need the data.

In [ ]:
# ---- Cleanup: Remove lab tables (uncomment to run) ----
# spark.sql("DROP TABLE IF EXISTS workspace.genai_labs.conversation_buffer")
# spark.sql("DROP TABLE IF EXISTS workspace.genai_labs.conversation_summaries")
# spark.sql("DROP TABLE IF EXISTS workspace.genai_labs.structured_memory")
# spark.sql("DROP TABLE IF EXISTS workspace.genai_labs.customer_profiles")
# spark.sql("DROP TABLE IF EXISTS workspace.genai_labs.agent_checkpoints")
# print("All lab tables dropped.")

---

## Next Lab

Continue to **Lab 04-05: Batch Inference with AI Query** (`05_batch_inference_ai_query.sql`), where you will learn how to run LLM inference at scale over entire Delta tables using Databricks SQL's `AI_QUERY()` function — no Python required.